## GPU check and installs

In [ ]:
!nvidia-smi
!pip install -q -U transformers accelerate datasets sqlparse nltk tqdm

Tue Jun 30 23:32:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Get the Spider databases

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# One-time: put spider_data.zip in your Drive root, then:
!cp "/content/drive/MyDrive/spider_data.zip" /content/
!unzip -q /content/spider_data.zip -d /content/
!ls /content/spider_data        # expect: database/  tables.json  dev.json  train_spider.json ...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
database      README.txt     test_gold.sql     train_gold.sql
dev_gold.sql  tables.json    test.json	       train_others.json
dev.json      test_database  test_tables.json  train_spider.json


## Imports and config

In [ ]:
import os, re, json, time, sqlite3, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SPIDER   = "/content/spider_data"
DB_DIR   = f"{SPIDER}/database"
TABLES   = f"{SPIDER}/tables.json"
DEV_JSON = f"{SPIDER}/dev.json"
DEVICE   = "cuda"

## Load the model

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
).eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## The verifier

In [ ]:
FORBIDDEN = ("drop","delete","update","insert","alter","create","replace","attach","pragma")

def build_schema_string(db_path, max_tables=20):
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True); cur = con.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    tables = [r[0] for r in cur.fetchall()][:max_tables]
    out = []
    for t in tables:
        cur.execute(f'PRAGMA table_info("{t}")')
        out.append(f"{t}({', '.join(r[1] for r in cur.fetchall())})")
    con.close(); return "\n".join(out)

def extract_sql(text):
    m = re.search(r"```sql\s*(.*?)```", text, re.DOTALL|re.IGNORECASE) or \
        re.search(r"```\s*(.*?)```", text, re.DOTALL)
    sql = (m.group(1) if m else text).strip().rstrip(";").strip()
    return re.sub(r"\s+", " ", sql)

def safe_execute(db_path, sql, timeout=5.0):
    if not sql or any(k in sql.lower() for k in FORBIDDEN): return None
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=timeout)
    start = time.time(); con.set_progress_handler(lambda: time.time()-start > timeout, 10000)
    try: return con.cursor().execute(sql).fetchall()
    except Exception: return None
    finally: con.close()

def exec_match(db_path, pred_sql, gold_sql):
    p = safe_execute(db_path, pred_sql)
    if p is None: return 0.0
    g = safe_execute(db_path, gold_sql)
    if g is None: return 0.0
    return 1.0 if set(map(tuple, p)) == set(map(tuple, g)) else 0.0

# sanity: a gold query must score 1.0
_d = json.load(open(DEV_JSON))[0]
_dbp = f"{DB_DIR}/{_d['db_id']}/{_d['db_id']}.sqlite"
assert exec_match(_dbp, _d["query"], _d["query"]) == 1.0, "Verifier broken — check DB_DIR"
print("Verifier OK")

Verifier OK


## Build the dev prompts

In [ ]:
PROMPT = ("Database schema:\n{schema}\n\nWrite a single SQLite query that answers "
          "the question. Return ONLY the query inside a ```sql code block.\n\nQuestion: {question}")

dev = json.load(open(DEV_JSON))
examples = []
for ex in tqdm(dev):
    dbp = f"{DB_DIR}/{ex['db_id']}/{ex['db_id']}.sqlite"
    msg = [{"role":"user","content": PROMPT.format(
        schema=build_schema_string(dbp), question=ex["question"])}]
    text = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    examples.append({"text": text, "gold": ex["query"], "db_id": ex["db_id"], "db_path": dbp})
print(len(examples), "dev examples")

  0%|          | 0/1034 [00:00<?, ?it/s]

1034 dev examples


## Batched generation helper, then the greedy pass

In [ ]:
def generate_batched(texts, max_new=256, do_sample=False, num_return=1, temp=0.8, bs=16):
    outs = []
    for i in tqdm(range(0, len(texts), bs)):
        enc = tok(texts[i:i+bs], return_tensors="pt", padding=True,
                  truncation=True, max_length=1536).to(DEVICE)
        kw = dict(max_new_tokens=max_new, pad_token_id=tok.pad_token_id,
                  num_return_sequences=num_return)
        kw.update(dict(do_sample=True, temperature=temp, top_p=0.95) if do_sample else dict(do_sample=False))
        with torch.no_grad():
            g = model.generate(**enc, **kw)
        g = g[:, enc["input_ids"].shape[1]:]
        outs.extend(tok.batch_decode(g, skip_special_tokens=True))
    return outs

greedy = generate_batched([e["text"] for e in examples], do_sample=False, bs=16)
preds  = [extract_sql(g) or "SELECT 1" for g in greedy]

  0%|          | 0/65 [00:00<?, ?it/s]

## Write the prediction and gold files (aligned, one line each)

In [ ]:
with open("/content/gold.txt","w") as gf, open("/content/pred.txt","w") as pf:
    for e, p in zip(examples, preds):
        gf.write(f"{e['gold']}\t{e['db_id']}\n")
        pf.write(f"{p}\n")
print("wrote gold.txt and pred.txt")

wrote gold.txt and pred.txt


## pass@1 vs pass@8 diagnostic (subset, ~200 for speed)

In [ ]:
K, SUB = 8, 200
sub = examples[:SUB]
samp = generate_batched([e["text"] for e in sub], do_sample=True, num_return=K, temp=0.8, bs=8)
p1 = pk = 0.0
for i, e in enumerate(sub):
    cands = samp[i*K:(i+1)*K]
    p1 += exec_match(e["db_path"], preds[i], e["gold"])
    pk += 1.0 if any(exec_match(e["db_path"], extract_sql(c), e["gold"]) >= 1 for c in cands) else 0.0
print(f"pass@1 (greedy): {p1/len(sub):.3f}")
print(f"pass@{K}        : {pk/len(sub):.3f}")

  0%|          | 0/25 [00:00<?, ?it/s]

pass@1 (greedy): 0.450
pass@8        : 0.660


## Clone the official evaluators

In [ ]:
!git clone -q https://github.com/taoyds/spider
!git clone -q https://github.com/taoyds/test-suite-sql-eval

## EX: the reportable execution accuracy + exact match, by difficulty

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
!cd spider && python evaluation.py --gold /content/gold.txt --pred /content/pred.txt --db {DB_DIR} --table {TABLES} --etype all

eval_err_num:1
medium pred: SELECT AVG(Age) AS Average_Age, MIN(Age) AS Minimum_Age, MAX(Age) AS Maximum_Age FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:2
medium pred: SELECT AVG(Age) AS Average_Age, MIN(Age) AS Minimum_Age, MAX(Age) AS Maximum_Age FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:3
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID ORDER BY T1.Age ASC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:4
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer)
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:

## TS: test-suite accuracy (do this after EX works)

In [ ]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
# After placing test_suite_database.zip in your Drive root:
!cp "/content/drive/MyDrive/test_suite_database.zip" /content/
!unzip -q /content/test_suite_database.zip -d /content/
!ls /content/test_suite_database | head     # folder of <db_id>/<db_id>.sqlite, like the regular DBs

replace /content/database/voter_1/voter_1v515patch1.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
ls: cannot access '/content/test_suite_database': No such file or directory


In [ ]:
!unzip -q /content/test_suite_database.zip -d /content/
!ls /content/test_suite_database | head

replace /content/database/voter_1/voter_1v515patch1.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/__MACOSX/database/voter_1/._voter_1v515patch1.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/database/voter_1/voter_1v515nightpatch3.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/database/voter_1/voter_1v515nightpatch3.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
ls: cannot access '/content/test_suite_database': No such file or directory


In [ ]:
!cd test-suite-sql-eval && python evaluation.py --gold /content/gold.txt --pred /content/pred.txt --db database --table {TABLES} --etype exec

Traceback (most recent call last):
  File "/content/test-suite-sql-eval/evaluation.py", line 938, in <module>
    evaluate(args.gold, args.pred, args.db, args.etype, kmaps, args.plug_value, args.keep_distinct, args.progress_bar_for_each_datapoint)
  File "/content/test-suite-sql-eval/evaluation.py", line 573, in evaluate
    schema = Schema(get_schema(db))
                    ^^^^^^^^^^^^^^
  File "/content/test-suite-sql-eval/process_sql.py", line 88, in get_schema
    conn = sqlite3.connect(db)
           ^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: unable to open database file


## Save everything to Drive

In [ ]:
import shutil, json, datetime

# timestamped folder so reruns don't overwrite
stamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = f"/content/drive/MyDrive/spider_baseline/{stamp}"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) the prediction + gold files (needed to re-score anytime, no GPU required)
for f in ["/content/gold.txt", "/content/pred.txt"]:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)

# 2) the raw greedy generations (so you never have to regenerate)
with open(f"{SAVE_DIR}/raw_greedy.json", "w") as fh:
    json.dump(greedy, fh)

# 3) the pass@k diagnostic numbers + run config
summary = {
    "model": MODEL_ID,
    "n_dev": len(examples),
    "pass@1_greedy": round(p1/len(sub), 4),
    "pass@8":        round(pk/len(sub), 4),
    "passk_subset":  len(sub),
    "timestamp": stamp,
    "note": "EX/TS come from the official evaluators — paste their printed numbers below",
    "EX_by_difficulty": "FILL IN from cell 11 output",
    "TS_by_difficulty": "FILL IN from cell 12 output",
}
with open(f"{SAVE_DIR}/summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)

print("Saved to:", SAVE_DIR)
print(os.listdir(SAVE_DIR))

Saved to: /content/drive/MyDrive/spider_baseline/20260630_235525
['gold.txt', 'pred.txt', 'raw_greedy.json', 'summary.json']
